# Week 9 — Operational ML: Predictive Maintenance (Equipment Failure Prediction)

**Author:** _[Your Name]_
**Course:** Applied ML for Operations — Week 9

This notebook builds an end-to-end operational machine learning pipeline:
problem framing → data prep & imbalance handling → model training with
cross-validation → evaluation beyond accuracy → clustering (bonus) →
explainability (feature importance + SHAP).

Every major decision is commented with the **operational reasoning** behind
it, not just the code mechanics — that reasoning is the actual deliverable
for an operations audience.


## 1. Problem Definition

**Operational problem:** Unplanned equipment failure on a production line.

Unplanned downtime is expensive (lost throughput, rush repairs, safety
risk), while *planned* maintenance is comparatively cheap. The goal is to
give the maintenance team a **daily risk score per machine** so they can
schedule inspections for the highest-risk units before they fail, instead
of reacting after a breakdown.

**Target variable:** `failure`
- `1` = the machine experiences a failure within the next maintenance
  window (e.g., the next 7 operating days)
- `0` = the machine keeps running normally through that window

**Unit of analysis:** one row = one machine-day snapshot of sensor and
usage readings.

**Why this framing matters operationally:** this is a *binary
classification* problem with a **rare positive class** (failures are, by
definition, uncommon) and **asymmetric costs**:
- A **false negative** (missed failure) → unplanned downtime, potential
  safety incident, expensive emergency repair.
- A **false positive** (false alarm) → a technician spends 20 minutes
  inspecting a machine that was fine.

Because a missed failure is far costlier than a false alarm, the model
should be tuned and evaluated to **favor recall on the failure class**,
not overall accuracy. This drives every choice below.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# NOTE: if a package below is missing in your environment, uncomment the
# install line once, then re-run. (Not needed on Colab — these ship by default.)
# !pip install xgboost shap imbalanced-learn --quiet


## 2. Data Preparation & Imbalance Check

For this notebook we **synthesize a realistic sensor dataset** so the
pipeline is fully self-contained and reproducible. In production you would
replace the block below with a `pd.read_csv(...)` / SQL pull from your
historian / SCADA / CMMS system — the rest of the pipeline (imbalance
handling, modeling, evaluation, explainability) does not need to change.

Features are the kind of signals a real predictive-maintenance system
would have: temperature, vibration, pressure, rotation speed, tool wear,
humidity, cumulative operating hours, and recent error/alarm counts.


In [ ]:
# --- Synthetic operational dataset -----------------------------------------
# In production, replace this cell with: df = pd.read_csv("machine_sensor_log.csv")
n_samples = 6000

temperature       = np.random.normal(75, 10, n_samples)     # deg C
vibration         = np.random.normal(2.5, 0.8, n_samples)   # mm/s RMS
pressure          = np.random.normal(100, 15, n_samples)    # psi
rotation_speed    = np.random.normal(1500, 200, n_samples)  # rpm
tool_wear         = np.random.uniform(0, 250, n_samples)    # minutes of wear
humidity          = np.random.normal(45, 10, n_samples)     # %
operating_hours   = np.random.uniform(0, 10000, n_samples)  # cumulative hours
error_count_30d   = np.random.poisson(1.2, n_samples)       # alarms in last 30 days

# Ground-truth generating process: vibration and recent errors dominate risk,
# with smaller contributions from temperature, tool wear and run time.
# This mirrors real predictive-maintenance domain knowledge (vibration is
# usually the earliest and strongest indicator of mechanical degradation).
risk_score = (
    0.04 * (temperature - 75)
    + 1.80 * (vibration - 2.5)
    + 0.02 * (pressure - 100)
    + 0.01 * (tool_wear - 125) / 10
    + 0.0003 * operating_hours
    + 0.50 * error_count_30d
    + np.random.normal(0, 2, n_samples)          # irreducible noise
)
prob_failure = 1 / (1 + np.exp(-(risk_score - 6)))
failure = (np.random.rand(n_samples) < prob_failure).astype(int)

df = pd.DataFrame({
    "temperature": temperature,
    "vibration": vibration,
    "pressure": pressure,
    "rotation_speed": rotation_speed,
    "tool_wear": tool_wear,
    "humidity": humidity,
    "operating_hours": operating_hours,
    "error_count_30d": error_count_30d,
    "failure": failure,
})

print(df.shape)
df.head()


In [ ]:
# --- Class distribution / imbalance check -----------------------------------
class_counts = df["failure"].value_counts()
class_pct = df["failure"].value_counts(normalize=True) * 100
print("Class counts:\n", class_counts)
print("\nClass percentages:\n", class_pct.round(2))

fig, ax = plt.subplots()
sns.countplot(x="failure", data=df, ax=ax, palette=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["No Failure (0)", "Failure (1)"])
ax.set_title("Class Distribution: Equipment Failure")
for p in ax.patches:
    ax.annotate(f"{p.get_height():,}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()

# WHY THIS CHECK MATTERS:
# Failures are ~10-12% of records here, which is a classic moderate-to-severe
# imbalance for an operational dataset. A model that always predicts "no
# failure" would already score ~88-90% accuracy while being operationally
# useless (it would never catch a single failure). This is exactly why we
# check the distribution *before* choosing metrics and *before* training.


### Handling the imbalance

Two standard techniques address this, and we demonstrate both so the
tradeoff is explicit:

1. **Class weights** (`class_weight="balanced"` / `scale_pos_weight`) — the
   model penalizes mistakes on the minority (failure) class more heavily
   during training. Cheap, no synthetic data, and works well with
   tree ensembles.
2. **SMOTE** (Synthetic Minority Oversampling Technique) — generates
   synthetic minority-class examples by interpolating between real
   failure cases, rebalancing the *training* set.

**Operational choice:** we use **class weights** as the primary approach
for Random Forest and XGBoost (both support it natively, it's fast, and it
avoids fabricating synthetic sensor readings that don't correspond to a
physically real machine state). We also show **SMOTE inside a proper
pipeline** as an alternative/comparison, applied *only to the training
fold* to avoid data leakage into validation/test.


In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split preserves the failure rate in both train and test sets —
# critical for imbalanced data, otherwise the test set could end up with
# almost no positive examples and evaluation would be meaningless.
X = df.drop(columns="failure")
y = df["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train failure rate:", y_train.mean().round(3))
print("Test failure rate: ", y_test.mean().round(3))


In [ ]:
# --- SMOTE demonstration (train-fold only, to avoid leakage) ---------------
try:
    from imblearn.over_sampling import SMOTE

    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    print("Before SMOTE:", y_train.value_counts().to_dict())
    print("After SMOTE: ", y_train_smote.value_counts().to_dict())
except ImportError:
    print("imbalanced-learn not installed — run `pip install imbalanced-learn` "
          "to try the SMOTE comparison. Proceeding with class-weighting only.")
    X_train_smote, y_train_smote = X_train, y_train


## 3. Model Training

We train **two model families**, both tree ensembles, which is the
standard choice for tabular operational data because they:
- handle mixed-scale numeric sensor features without scaling,
- capture non-linear interactions (e.g., high vibration only matters at
  high operating hours),
- give us feature importances and are SHAP-compatible for explainability.

**Random Forest** — bagging ensemble, robust, low variance, easy to
explain to non-technical stakeholders ("a committee of decision trees
voting").

**XGBoost** — boosting ensemble, typically higher predictive performance
on structured/tabular data, and natively supports `scale_pos_weight` for
imbalance.

We validate both with **Stratified K-Fold cross-validation** (not a single
train/test split) so the reported performance reflects the model's
stability across different subsets of machines, not a lucky/unlucky split.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    class_weight="balanced",   # <-- imbalance handling: upweight the rare failure class
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# We score CV on multiple metrics at once because, as noted in Section 1,
# accuracy alone would hide poor minority-class performance.
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
rf_cv_results = cross_validate(rf_model, X_train, y_train, cv=cv, scoring=scoring)

print("Random Forest — 5-fold Stratified CV (train set)")
for metric in scoring:
    scores = rf_cv_results[f"test_{metric}"]
    print(f"  {metric:9s}: mean={scores.mean():.3f}  std={scores.std():.3f}")


In [ ]:
try:
    from xgboost import XGBClassifier

    # scale_pos_weight = (# negative) / (# positive) is XGBoost's native
    # equivalent of class_weight="balanced" — it tells the loss function to
    # penalize a missed failure roughly `scale_pos_weight` times more than
    # a false alarm.
    neg, pos = np.bincount(y_train)
    scale_pos_weight = neg / pos

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    xgb_cv_results = cross_validate(xgb_model, X_train, y_train, cv=cv, scoring=scoring)
    print("XGBoost — 5-fold Stratified CV (train set)")
    for metric in scoring:
        scores = xgb_cv_results[f"test_{metric}"]
        print(f"  {metric:9s}: mean={scores.mean():.3f}  std={scores.std():.3f}")
    HAS_XGB = True
except ImportError:
    print("xgboost not installed — run `pip install xgboost`. "
          "Falling back to GradientBoostingClassifier so the notebook still runs end-to-end.")
    from sklearn.ensemble import GradientBoostingClassifier
    xgb_model = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                            random_state=RANDOM_STATE)
    xgb_cv_results = cross_validate(xgb_model, X_train, y_train, cv=cv, scoring=scoring)
    print("GradientBoostingClassifier (XGBoost fallback) — 5-fold Stratified CV")
    for metric in scoring:
        scores = xgb_cv_results[f"test_{metric}"]
        print(f"  {metric:9s}: mean={scores.mean():.3f}  std={scores.std():.3f}")
    HAS_XGB = False


In [ ]:
# --- Fit final models on the full training set for downstream evaluation ---
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
print("Both models fitted on the full training set.")


## 4. Evaluation — Beyond Accuracy

**Why we do NOT rely on accuracy alone:** with an ~88-90% majority class, a
trivial "always predict no-failure" model scores ~88-90% accuracy while
catching zero real failures. For an operational deployment, that's a
silent, dangerous failure mode of the *evaluation*, not just the model.

Instead we report:
- **Confusion matrix** — the actual counts operations cares about: how
  many failures did we catch (true positives) vs. miss (false negatives),
  and how many false alarms (false positives) did we generate.
- **Precision** — of the machines we flagged as high-risk, how many
  actually failed? (Controls technician time wasted on false alarms.)
- **Recall** — of the machines that actually failed, how many did we
  catch? (This is the metric that matters most given our cost asymmetry.)
- **F1-score** — harmonic mean of precision/recall, a single balanced
  summary number.
- **ROC-AUC** — threshold-independent measure of how well the model ranks
  failing machines above healthy ones; useful because in practice we can
  tune the decision threshold to trade precision for recall depending on
  technician capacity.


In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay,
)

def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"===== {name} — Test Set Evaluation =====")
    print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Confusion matrix — the numbers an ops manager actually reads
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No Failure", "Failure"]).plot(
        ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title(f"{name}: Confusion Matrix")

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba):.3f}")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate (Recall)")
    axes[1].set_title(f"{name}: ROC Curve")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return {
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }

rf_metrics = evaluate_model(rf_model, X_test, y_test, "Random Forest")


In [ ]:
xgb_name = "XGBoost" if HAS_XGB else "GradientBoosting (XGBoost fallback)"
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, xgb_name)


In [ ]:
# --- Side-by-side comparison -------------------------------------------------
comparison = pd.DataFrame([rf_metrics, xgb_metrics], index=["Random Forest", xgb_name])
print(comparison.round(3))

# Interpretation guidance for the write-up:
# - Pick the model with the higher RECALL at an acceptable precision, since
#   missed failures are the costlier error for this operational context.
# - If technician capacity is very limited, precision matters more and the
#   decision threshold (not just the model) should be tuned accordingly —
#   ROC-AUC tells us the model's ranking quality independent of that choice.


## 5. Clustering (Optional Bonus) — Machine Risk Segments

Beyond a single failure/no-failure prediction, **K-Means clustering** lets
us segment machines into operational profiles (e.g., "high-usage,
high-vibration", "new/low-wear", "aging but stable") — useful for planning
maintenance *strategy*, not just reacting to individual risk scores.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Scale features first — K-Means is distance-based, so unscaled features
# like operating_hours (0-10,000) would completely dominate vibration (0-5).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow method + silhouette score to pick a reasonable k
inertias, sil_scores = [], []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_range), sil_scores, marker="o", color="orange")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Score")
plt.tight_layout()
plt.show()


In [ ]:
# k=4 chosen as a reasonable balance of interpretability and separation
# (adjust based on the elbow/silhouette plots above for your real data).
k_final = 4
kmeans = KMeans(n_clusters=k_final, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Profile each cluster: average sensor readings + failure rate.
# This tells operations WHICH kind of machine profile each segment represents,
# and whether that segment is inherently higher-risk.
cluster_profile = df.groupby("cluster")[list(X.columns) + ["failure"]].mean().round(2)
cluster_profile["n_machines"] = df["cluster"].value_counts().sort_index()
cluster_profile


In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(data=df, x="vibration", y="operating_hours", hue="cluster",
                 palette="tab10", alpha=0.6, ax=ax)
ax.set_title("Machine Segments: Vibration vs. Operating Hours")
plt.show()

# Operational read: clusters with both high average vibration AND a high
# failure rate (see cluster_profile above) are the segment maintenance
# planning should prioritize for a proactive inspection schedule — this
# complements the per-machine model score with a fleet-level view.


## 6. Explainability — Feature Importance & SHAP

A model that flags "this machine is high risk" is not useful to a
maintenance technician unless it also explains **why**. We use two
complementary views:

- **Feature importance** (global) — which features matter most to the
  model *on average, across all machines*. Good for a one-time report to
  management on what drives failures fleet-wide.
- **SHAP values** (local + global) — how much each feature pushed *this
  specific machine's* prediction up or down from the baseline. This is
  what lets a technician look at one flagged machine and see "vibration
  and error count are why this one was flagged," which is the entire
  basis of the video walkthrough in Part B.


In [ ]:
# --- Global feature importance (Random Forest) ------------------------------
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots()
importances.plot(kind="barh", ax=ax, color="#4C72B0")
ax.invert_yaxis()
ax.set_title("Random Forest — Feature Importance")
ax.set_xlabel("Relative Importance")
plt.tight_layout()
plt.show()

print(importances.round(3))


In [ ]:
try:
    from xgboost import XGBClassifier as _XGBClassifier
    xgb_is_native = isinstance(xgb_model, _XGBClassifier)
except ImportError:
    xgb_is_native = False

xgb_importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots()
xgb_importances.plot(kind="barh", ax=ax, color="#DD8452")
ax.invert_yaxis()
ax.set_title(f"{xgb_name} — Feature Importance")
ax.set_xlabel("Relative Importance")
plt.tight_layout()
plt.show()


In [ ]:
# --- SHAP: global summary + local explanation for one flagged machine ------
try:
    import shap

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test)

    # For a binary classifier, shap_values may be a list [class0, class1]
    # or a single array depending on the shap/sklearn version — handle both.
    sv_failure = shap_values[1] if isinstance(shap_values, list) else shap_values

    # Global view: which features matter, and in which direction, across all
    # test-set machines.
    shap.summary_plot(sv_failure, X_test, show=False)
    plt.title("SHAP Summary — Random Forest (Failure class)")
    plt.tight_layout()
    plt.show()

    HAS_SHAP = True
except ImportError:
    print("shap not installed — run `pip install shap` to generate the summary "
          "and force plots used in the Part B video walkthrough.")
    HAS_SHAP = False


In [ ]:
if HAS_SHAP:
    # --- Local explanation: pick the single highest-risk machine in the test
    # set that the model flagged, and explain that ONE prediction. This is
    # exactly the plot to narrate in the Part B video:
    # "Here is a case the model flagged as High Risk. Vibration was the main driver..."
    test_probs = rf_model.predict_proba(X_test)[:, 1]
    top_risk_idx_pos = np.argmax(test_probs)          # position within X_test
    top_risk_row = X_test.iloc[[top_risk_idx_pos]]

    print("Flagged machine — sensor readings:")
    print(top_risk_row)
    print(f"\nModel predicted failure probability: {test_probs[top_risk_idx_pos]:.2%}")

    shap.force_plot(
        explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray))
            else explainer.expected_value,
        sv_failure[top_risk_idx_pos],
        top_risk_row,
        matplotlib=True,
        show=False,
    )
    plt.title("SHAP Force Plot — Highest-Risk Flagged Machine")
    plt.tight_layout()
    plt.show()

    # Bar-style local explanation (often clearer on screen recordings than the
    # force plot) — shows each feature's contribution to THIS prediction.
    shap.plots._waterfall.waterfall_legacy(
        explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray))
            else explainer.expected_value,
        sv_failure[top_risk_idx_pos],
        feature_names=list(X_test.columns),
        show=False,
    )
    plt.title("SHAP Waterfall — Highest-Risk Flagged Machine")
    plt.tight_layout()
    plt.show()


**Reading the plots for the video walkthrough:** in the SHAP summary plot,
each dot is one machine; red = high feature value, blue = low; position
left/right shows whether that value pushed the prediction toward "failure"
or "no failure." In the force/waterfall plot for the single flagged
machine, each bar shows how much that feature's specific value moved the
prediction away from the average baseline — this is the "Vibration was the
main driver" narration moment.


## 7. Summary of Modeling Decisions (Documentation)

| Decision | Choice | Operational Reasoning |
|---|---|---|
| Target variable | `failure` within next maintenance window | Actionable time horizon technicians can plan around |
| Imbalance handling | Class weights (primary), SMOTE (compared) | Avoids fabricating unrealistic sensor states; both tested for robustness |
| Validation | Stratified 5-fold CV | Preserves rare failure class in every fold; avoids an unlucky single split |
| Models | Random Forest + XGBoost | Both handle mixed-scale tabular sensor data, support class weighting, and are SHAP-compatible |
| Primary metric | Recall (with precision/F1/ROC-AUC reported) | Missed failures are far costlier than false alarms in this context |
| Explainability | Feature importance (global) + SHAP (local) | Global view for management reporting; local view for technician trust on individual flagged machines |
| Clustering | K-Means (k chosen via elbow/silhouette) | Segments the fleet for proactive maintenance strategy, complementing per-machine scores |

**Next steps for a real deployment:** replace the synthetic data generator
in Section 2 with a live feed from the historian/CMMS, re-validate the
decision threshold against actual technician capacity, and monitor for
model drift as equipment ages or maintenance practices change.
